In [2]:
pip install gensim pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 66.2 MB/s eta 0:00:00


In [4]:
# ============================================================
# This script trains the Word2Vec models in two architectures.
# (CBOW and Skip-gram) on various hyperparameters,
# then stores results and the most successful models.
# ============================================================

import os          # To construct file paths and deal with directories
import glob        # In the case of pattern-based file discovery (e.g., *.txt)
import zipfile     # Unpacking the corpus uploaded
import pandas as pd                  # To tabulate the results of an experiment
from gensim.models import Word2Vec   # Word embedding training core library
from google.colab import files       # Colab file upload/downloading utility

# ------------------------------------------------------------
# STEP 1: Load and Tokenize the Corpus File
# ------------------------------------------------------------
def load_corpus(file_path="corpus.txt"):
    """
    Reads the single integrated corpus file and prepares it for
    Word2Vec training by tokenizing each line.

    Args:
        file_path (str): The path to the corpus.txt file.

    Returns:
        sentences (list of list of str): A list where each element is a
                                         list of tokens from one line/page.
    """
    print(f"Reading corpus from '{file_path}'...")
    sentences = []

    # Check if the specific file exists to prevent silent crashes
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file '{file_path}' was not found. Please run the scraper first.")

    with open(file_path, 'r', encoding='utf-8') as f:
        # Read the file line by line (assuming each line is a different page/section)
        for line in f:
            # Basic cleaning: lowercase the text
            content = line.strip().lower()

            if not content:
                continue  # Skip empty lines

            # Tokenize by whitespace and filter out tokens with length <= 2
            # This helps remove noise and non-semantic characters
            tokens = [word for word in content.split() if len(word) > 2]

            # Word2Vec expects a list of tokenized sentences/documents
            if tokens:
                sentences.append(tokens)

    print(f"✅ Successfully loaded {len(sentences)} documents/sentences for training.")
    return sentences


# ------------------------------------------------------------
# STEP 2: Run Hyperparameter Experiments
# ------------------------------------------------------------
def run_experiments(sentences):
    """
    Trains Word2Vec models across all combinations of:
      - Architectures: CBOW (sg=0) and Skip-gram (sg=1)
      - Vector dimensions: 50, 100, 200
      - Context window sizes: 2, 5, 8
      - Negative samples: 5, 10, 15

    That gives 2 × 3 × 3 × 3 = 54 total training runs.

    The models trained with dimension=100, window=5, neg=5
    are designated as the "best" and saved to disk.

    Args:
        sentences (list of list of str): Tokenized corpus from load_corpus().

    Returns:
        Tuple of (pd.DataFrame of all results, best Skip-gram model)
    """

    # Hyperparameter space definition
    dimensions      = [50, 100, 200]   # Embedding vector size
    windows         = [2, 5, 8]        # Context words surrounding the word length
    negative_samples = [5, 10, 15]     # Negatively sampled words used in training

    results_list = []   # will record one dict at a time experimen
    best_skipgram = None  # Generator of saved Skip-gram model
    best_cbow     = None  # Save the saved CBOW model here

    print("Starting Experiments (CBOW vs Skip-gram)...")

    # Outer loop: sg=0 trains CBOW, sg=1 trains Skip-gram.
    for sg_mode in [0, 1]:
        arch_name = "Skip-gram" if sg_mode == 1 else "CBOW"

        for d in dimensions:
            for w in windows:
                for neg in negative_samples:

                    # Train a Word2Vec model with the given hyperparameter set
                    model = Word2Vec(
                        sentences=sentences,   # Tokenized corpus
                        vector_size=d,         # Dimensionality of word vectors
                        window=w,              # Context window size
                        negative=neg,          # Training sample noise words
                        sg=sg_mode,            # 0 = CBOW, 1 = Skip-gram
                        min_count=1,           # Include words that are used at least one time
                        epochs=30              # Whole passages through the corpus in training
                    )

                    # Note the important information of this run to include in the final report
                    results_list.append({
                        "Architecture": arch_name,
                        "Dimension":    d,
                        "Window":       w,
                        "Neg_Samples":  neg,
                        "Vocab_Size":   len(model.wv)  # Unique words learned
                    })

                    # Save the (d=100, w=5, neg=5) reference models to be analyzed later
                    if sg_mode == 1 and d == 100 and w == 5 and neg == 5:
                        best_skipgram = model
                    if sg_mode == 0 and d == 100 and w == 5 and neg == 5:
                        best_cbow = model

    # Safety check - it must be put in place as long as the grid remains the same
    if best_skipgram is None or best_cbow is None:
        raise RuntimeError("Best models not found — check hyperparameter grid.")

    # Store persist models to disk as binary format in Gensim
    best_skipgram.save("iitj_skipgram.model")
    best_cbow.save("iitj_cbow.model")

    # Send the results table and the Skip-gram model back upstream
    return pd.DataFrame(results_list), best_skipgram


# ------------------------------------------------------------
# MAIN EXECUTION - executes when this script is indirectly invoked
# ------------------------------------------------------------
if __name__ == "__main__":
    try:
        # 1. Import the corpus to the Colab runtime
        # handle_upload()

        # 2. Read and tokenize any text files into a list of word lists
        corpus = load_corpus("corpus.txt")

        # 3. Run all of 54 training experiments and access the results
        report_df, final_model = run_experiments(corpus)

        # 4. Show a table of results in the notebook
        print("\n" + "=" * 60)
        print("           FORMAL MODEL TRAINING REPORT")
        print("=" * 60)
        print(report_df.to_string(index=False))
        print("=" * 60)

        # 5. export the results table when ready as a CSV
        report_df.to_csv("training_experiments_report.csv", index=False)
        print("Report saved as: training_experiments_report.csv")

        # 6. Export all output files of the Colab environment
        files.download("training_experiments_report.csv")
        files.download("iitj_skipgram.model")
        files.download("iitj_cbow.model")

    except Exception as e:
        # Catch-all handler - does not crash silently and prints an ambiguous error message
        print(f"Error during Task 2: {e}")

Reading corpus from 'corpus.txt'...
✅ Successfully loaded 3182 documents/sentences for training.
Starting Experiments (CBOW vs Skip-gram)...

           FORMAL MODEL TRAINING REPORT
Architecture  Dimension  Window  Neg_Samples  Vocab_Size
        CBOW         50       2            5        7990
        CBOW         50       2           10        7990
        CBOW         50       2           15        7990
        CBOW         50       5            5        7990
        CBOW         50       5           10        7990
        CBOW         50       5           15        7990
        CBOW         50       8            5        7990
        CBOW         50       8           10        7990
        CBOW         50       8           15        7990
        CBOW        100       2            5        7990
        CBOW        100       2           10        7990
        CBOW        100       2           15        7990
        CBOW        100       5            5        7990
        CBOW        

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>